# Qwen3-0.6B LoRA: combined private dialogue experiment

This notebook trains a LoRA adapter for the same Qwen3-0.6B base model used by the fast local Q4_K_M GGUF. It combines 2,480 synthetic Turkish Daily Dialogues records (CC BY 4.0, draft/unreviewed individually) with the 20 private project-authored dialogues (owner-reviewed, not for public redistribution). The 20 private records are kept local to Kaggle and are never included in this public GitHub repository. Group-safe splitting keeps the private examples in train; the public source supplies validation and held-out test data.

The notebook downloads the code and public licensed data from GitHub, and automatically finds the separately attached private Kaggle Dataset by its fixed ZIP filename. Upload `artifacts/qwen3_private_dialogue_addon_001.zip` as a **Private** Kaggle Dataset once and attach it to this notebook; you do not need to select individual files. Then enable a GPU and internet access and run all cells. Do not publish the private training data or the resulting adapter.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

GITHUB_REPO_URL = 'https://github.com/dozagoat47-star/qwen3-turkish-lora-kaggle.git'
GITHUB_REF = 'qwen3-combined-private-001'
repo_root = Path('/kaggle/working/qwen3_lora_bundle')
if repo_root.exists():
    shutil.rmtree(repo_root)
subprocess.check_call([
    'git', 'clone', '--depth', '1', '--branch', GITHUB_REF,
    GITHUB_REPO_URL, str(repo_root),
])
bundle_path = repo_root / 'datasets/qwen3_0_6b_lora_bundle_002.zip'
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    for member in members:
        name = PurePosixPath(member.filename)
        if name.is_absolute() or '..' in name.parts:
            raise ValueError(f'Unsafe path in Kaggle bundle: {member.filename}')

    archive.extractall(repo_root)

bundle_manifest = json.loads((repo_root / 'bundle_manifest.json').read_text(encoding='utf-8'))
for relative_path, expected_hash in bundle_manifest['files'].items():
    actual_hash = hashlib.sha256((repo_root / relative_path).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'Bundle hash mismatch: {relative_path}')
if bundle_manifest['review_status'] != 'draft' or bundle_manifest['license'] != 'CC BY 4.0':
    raise ValueError('Unexpected dataset review status or license')

public_data_path = repo_root / 'data/raw/turkish_daily_dialogues_sft_001/train.jsonl'
sys.path.insert(0, str(repo_root))
from clean_model.data import dataset_fingerprint, load_jsonl, split_by_group

public_data_bytes = public_data_path.read_bytes()
if hashlib.sha256(public_data_bytes).hexdigest() != bundle_manifest['training_data_sha256']:
    raise ValueError('Public dialogue data hash does not match its bundle manifest')
public_records = load_jsonl(public_data_path, require_approved=False)
if len(public_records) != bundle_manifest['record_count']:
    raise ValueError('Public dialogue record count does not match the verified bundle')
if any(record['review']['status'] != 'draft' for record in public_records):
    raise ValueError('Expected the explicitly opted-in draft dialogue corpus')
if bundle_manifest['source_selection']['source_split'] != 'train only':
    raise ValueError('The bundle must contain only upstream train data')
print(f"Verified {len(public_records)} draft public conversations; license: {bundle_manifest['license']}")
print(bundle_manifest['attribution'])

private_archive_name = 'qwen3_private_dialogue_addon_001.zip'
private_archives = list(Path('/kaggle/input').rglob(private_archive_name))
if len(private_archives) > 1:
    raise RuntimeError('Attach only one Private Kaggle Dataset containing ' + private_archive_name)
if private_archives:
    with zipfile.ZipFile(private_archives[0]) as archive:
        if set(archive.namelist()) != {'manifest.json', 'qwen3_dialogue_review_001.jsonl'}:
            raise ValueError('Unexpected files in the private dialogue add-on')
        private_manifest = json.loads(archive.read('manifest.json').decode('utf-8'))
        private_data_bytes = archive.read('qwen3_dialogue_review_001.jsonl')
else:
    private_data_paths = list(Path('/kaggle/input').rglob('qwen3_dialogue_review_001.jsonl'))
    private_manifest_paths = list(Path('/kaggle/input').rglob('manifest.json'))
    pairs = [
        (data_file, data_file.parent / 'manifest.json')
        for data_file in private_data_paths
        if (data_file.parent / 'manifest.json') in private_manifest_paths
    ]
    if len(pairs) != 1:
        raise RuntimeError(
            'Attach one Private Kaggle Dataset containing '
            'qwen3_private_dialogue_addon_001.zip; found %d usable datasets' % len(pairs)
        )
    private_data_file, private_manifest_file = pairs[0]
    private_data_bytes = private_data_file.read_bytes()
    private_manifest = json.loads(private_manifest_file.read_text(encoding='utf-8'))
if private_manifest['review_status'] != 'approved' or private_manifest['record_count'] != 20:
    raise ValueError('Expected the 20 owner-approved private dialogue records')
if hashlib.sha256(private_data_bytes).hexdigest() != private_manifest['data_sha256']:
    raise ValueError('Private dialogue data SHA-256 does not match its manifest')
private_data_path = Path('/kaggle/working/qwen3_private_dialogue_review_001.jsonl')
private_data_path.write_bytes(private_data_bytes)
private_records = load_jsonl(private_data_path, require_approved=True)
if len(private_records) != 20 or any(
        record['review'].get('reviewed_by') != 'project_owner'
        for record in private_records):
    raise ValueError('Private dataset must contain 20 owner-approved records')
if dataset_fingerprint(private_records) != private_manifest['dataset_fingerprint']:
    raise ValueError('Private dialogue dataset fingerprint mismatch')
public_ids = {record['id'] for record in public_records}
private_ids = {record['id'] for record in private_records}
public_groups = {record['group_id'] for record in public_records}
private_groups = {record['group_id'] for record in private_records}
if public_ids & private_ids or public_groups & private_groups:
    raise ValueError('Public and private dialogue IDs/groups must be disjoint')
records = public_records + private_records
record_splits = split_by_group(records, seed=7, require_approved=False)
train_groups = {record['group_id'] for record in record_splits['train']}
if not private_groups <= train_groups:
    raise ValueError('All 20 owner-approved private conversations must be in train')
for split_name, split_records in record_splits.items():
    print(f"{split_name}: {len(split_records)} conversations")
print('Verified all 20 owner-approved private conversations are in train')
data_path = Path('/kaggle/working/qwen3_combined_training.jsonl')
data_path.write_text(
    ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in records),
    encoding='utf-8',
)
print(f'Verified private owner-approved conversations: {len(private_records)} (private data is not printed)')
print(f'Combined training records: {len(records)}')

subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'transformers>=5,<6', 'peft>=0.17,<1', 'accelerate>=1,<2',
    'huggingface-hub>=1.16,<2', 'safetensors>=0.4,<1',
])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before training')
print('GPU:', torch.cuda.get_device_name(0))

## Train and select by validation loss

The trainer explicitly opts in to the public draft records while accepting the 20 approved private examples. It keeps whole scenario families/conversations in one split. The owner-approved private examples are expected to fall in train; validation loss and one-shot held-out review use the separated public data. The output path is new so a previous notebook run cannot be overwritten.

In [ ]:
from clean_model.qwen_lora import train_adapter

run_dir = Path('/kaggle/working/qwen3_0_6b_lora_run_003')
training_metadata = train_adapter(
    dataset_path=data_path,
    output_dir=run_dir,
    epochs=4,
    batch_size=4,
    learning_rate=1e-4,
    patience=2,
    seed=7,
    require_approved=False,
)
print(json.dumps(training_metadata, ensure_ascii=False, indent=2))

## One-shot held-out comparison

Run this only once after training/settings are fixed. It compares the base and LoRA adapter on scenario families withheld from training and validation. Do not use this small qualitative report to repeatedly tune the run.

In [ ]:
import gc

evaluation_lock = run_dir / 'test_evaluation_started.lock'
with evaluation_lock.open('x', encoding='utf-8') as lock:
    lock.write('One-shot held-out evaluation started. Do not rerun for tuning.\n')
gc.collect()
torch.cuda.empty_cache()
from clean_model.qwen_lora import evaluate_test_once

test_report = evaluate_test_once(
    dataset_path=data_path,
    adapter_dir=run_dir,
    report_path=run_dir / 'one_shot_test_evaluation.json',
    seed=7,
    require_approved=False,
)
print(json.dumps(test_report, ensure_ascii=False, indent=2))

## Export a llama.cpp adapter

The previous run produced a valid safetensors adapter but did not complete this export cell: its metadata has no `gguf_adapter` record and no GGUF/ZIP output. This cell runs after training and held-out evaluation, verifies the pinned llama.cpp converter revision and resulting adapter file, records the GGUF SHA-256, then creates one downloadable ZIP. If export fails, Kaggle will show the exact failing command/error and the existing trained adapter remains in the output directory.

In [ ]:
from clean_model.qwen_lora import MODEL_ID, MODEL_REVISION
from transformers import AutoConfig, AutoTokenizer

llama_cpp_commit = 'f9a95aeb2096d0c0218e0cd6e6d901cd37be798a'
llama_cpp_dir = Path('/kaggle/working/llama.cpp')
if llama_cpp_dir.exists():
    shutil.rmtree(llama_cpp_dir)
subprocess.check_call([
    'git', 'clone', '--depth', '1', '--filter=blob:none', '--no-checkout',
    'https://github.com/ggml-org/llama.cpp.git', str(llama_cpp_dir),
])
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'fetch', '--depth', '1',
    'origin', llama_cpp_commit,
])
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'checkout', '--detach', 'FETCH_HEAD',
])
actual_llama_cpp_commit = subprocess.check_output(
    ['git', '-C', str(llama_cpp_dir), 'rev-parse', 'HEAD'], text=True,
).strip()
if actual_llama_cpp_commit != llama_cpp_commit:
    raise RuntimeError(f'Unexpected llama.cpp revision: {actual_llama_cpp_commit}')
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'sparse-checkout', 'init', '--no-cone',
])
subprocess.check_call([
    'git', '-C', str(llama_cpp_dir), 'sparse-checkout', 'set', '--no-cone',
    '/convert_lora_to_gguf.py', '/conversion/', '/gguf-py/',
])

base_config_dir = run_dir / 'base_model_config'
base_config_dir.mkdir(parents=True, exist_ok=True)
AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION).save_pretrained(base_config_dir)
AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, use_fast=True,
).save_pretrained(base_config_dir)
adapter_gguf_path = run_dir / 'adapter_model.gguf'
if adapter_gguf_path.exists():
    adapter_gguf_path.unlink()
subprocess.check_call([
    sys.executable,
    str(llama_cpp_dir / 'convert_lora_to_gguf.py'),
    '--outfile', str(adapter_gguf_path),
    '--outtype', 'f16',
    '--base', str(base_config_dir),
    str(run_dir),
])
if not adapter_gguf_path.is_file() or adapter_gguf_path.stat().st_size <= 1024:
    raise RuntimeError('GGUF converter returned without producing a valid adapter_model.gguf')
adapter_gguf_sha256 = hashlib.sha256(adapter_gguf_path.read_bytes()).hexdigest()
training_metadata['gguf_adapter'] = {
    'file': adapter_gguf_path.name,
    'sha256': adapter_gguf_sha256,
    'format': 'f16 LoRA GGUF',
    'llama_cpp_commit': actual_llama_cpp_commit,
    'base_model_revision': MODEL_REVISION,
}
(run_dir / 'training_metadata.json').write_text(
    json.dumps(training_metadata, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)
print('GGUF adapter:', adapter_gguf_path, 'bytes:', adapter_gguf_path.stat().st_size)
print('SHA-256:', adapter_gguf_sha256)
import zipfile

deliverables = [
    'adapter_model.gguf', 'adapter_model.safetensors', 'adapter_config.json',
    'training_metadata.json', 'one_shot_test_evaluation.json',
    'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
]
output_archive = Path('/kaggle/working/qwen3_0_6b_lora_run_003.zip')
if output_archive.exists():
    output_archive.unlink()
required_deliverables = {
    'adapter_model.gguf', 'adapter_model.safetensors', 'adapter_config.json',
    'training_metadata.json', 'one_shot_test_evaluation.json',
    'tokenizer.json', 'tokenizer_config.json',
}
missing = sorted(name for name in required_deliverables if not (run_dir / name).is_file())
if missing:
    raise FileNotFoundError(f'Missing required model output files: {missing}')
with zipfile.ZipFile(output_archive, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in deliverables:
        path = run_dir / filename
        if path.is_file():
            archive.write(path, arcname=filename)
with zipfile.ZipFile(output_archive) as archive:
    corrupt_member = archive.testzip()
    if corrupt_member is not None:
        raise RuntimeError(f'Output ZIP failed CRC validation: {corrupt_member}')
    archived_files = set(archive.namelist())
    missing_from_archive = sorted(required_deliverables - archived_files)
    if missing_from_archive:
        raise RuntimeError(f'Output ZIP is missing required files: {missing_from_archive}')
print('Ready-to-download model bundle:', output_archive)
print('Use its adapter_model.gguf with the existing local Q4_K_M base GGUF.')